In [1]:
import torch
from basicsr.models.archs.downsampling import (
    PixelUnshuffleDown,
    ConvStride2Down,
    FrequencyPreservedPooling,
)

x = torch.randn(2, 16, 128, 128)

for name, module in [
    ("pixel_unshuffle", PixelUnshuffleDown()),
    ("conv_stride2", ConvStride2Down(16, 64)),
    ("fp", FrequencyPreservedPooling()),
]:
    y = module(x)
    print(name, y.shape)

pixel_unshuffle torch.Size([2, 64, 64, 64])
conv_stride2 torch.Size([2, 64, 64, 64])
fp torch.Size([2, 64, 64, 64])


In [1]:
import torch
from basicsr.models.archs.upsampling import PixelShuffleUp, LCTCUp, FreqAvgUp

x = torch.randn(2, 64, 64, 64)

mods = [
    ("pixelshuffle", PixelShuffleUp(64, 32)),
    ("lctc7", LCTCUp(64, 32, large_kernel=7)),
    ("lctc11p3", LCTCUp(64, 32, large_kernel=11, small_kernel=3)),
    ("freqavgup", FreqAvgUp(64, 32, padding="constant")),
]

for name, mod in mods:
    y = mod(x)
    print(name, y.shape, y.dtype)

pixelshuffle torch.Size([2, 32, 128, 128]) torch.float32
lctc7 torch.Size([2, 32, 130, 130]) torch.float32
lctc11p3 torch.Size([2, 32, 130, 130]) torch.float32
freqavgup torch.Size([2, 32, 128, 128]) torch.float32


In [1]:
import torch
from basicsr.models.archs.NAFNet_arch import NAFNet

configs = [
    ("convstride2", "pixelshuffle"),
    ("convstride2", "lctc_11_3"),
    ("convstride2", "freqavgup"),
    ("pixelunshuffle", "pixelshuffle"),
    ("pixelunshuffle", "freqavgup"),
    ("fp", "pixelshuffle"),
    ("fp", "freqavgup"),
]

x = torch.randn(1, 3, 128, 128)

for down, up in configs:
    try:
        model = NAFNet(
            img_channel=3,
            width=8,
            middle_blk_num=1,
            enc_blk_nums=[1, 1],
            dec_blk_nums=[1, 1],
            downsample_type=down,
            upsample_type=up,
        )
        y = model(x)
        print(down, up, y.shape)
    except Exception as e:
        print(down, up, "ERROR:", repr(e))

convstride2 pixelshuffle torch.Size([1, 3, 128, 128])
convstride2 lctc_11_3 torch.Size([1, 3, 128, 128])
convstride2 freqavgup torch.Size([1, 3, 128, 128])
pixelunshuffle pixelshuffle torch.Size([1, 3, 128, 128])
pixelunshuffle freqavgup torch.Size([1, 3, 128, 128])
fp pixelshuffle torch.Size([1, 3, 128, 128])
fp freqavgup torch.Size([1, 3, 128, 128])
